# Download data — chapter 1

This notebook downloads the source datasets used by `slavic-speech-pipeline` from the CLARIN.SI repository.

**What it does**

- Curl-style download of one or more datasets (ROG, GOS, ParlaSpeech) into `data/raw/<dataset>/`.
- Unpack archives into `data/unpacked/<dataset>/`.
- Idempotent: skips files that already exist and are non-empty.
- Refuses to download multi-GB files unless `confirm_large=True`.

**What it does *not* do**

- Convert anything into the canonical JSONL — that's `prep_<dataset>.ipynb`'s job.
- Cut WAVs — that's `audio_splitter.py`'s job.
- Manage CLARIN authentication for restricted resources (e.g. GOS audio). Those need a manual step.

---

## 0. Imports and project root setup


In [1]:
# Make sure we can import utils_dataprep no matter where Jupyter was launched.
import sys
from pathlib import Path

# Find the chapter folder this notebook lives in, then walk up to project root.
HERE = Path.cwd()
if HERE.name != "1_data_prep":
    # If launched from repo root, point at the chapter folder
    candidate = HERE / "1_data_prep"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE))

import utils_dataprep as udp
PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Chapter dir = {HERE}")


PROJECT_ROOT = /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline
Chapter dir = /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/1_data_prep


---

## 1. Config

The dataset registry below is the single source of truth for what we can download. Each entry lists:

- `handle` — the CLARIN handle (just for human reference)
- `base_url` — the CLARIN bitstream URL prefix
- `files` — list of `(filename, approximate_size_mb, is_large)` tuples
- `notes` — anything quirky about this dataset

To run the notebook: set `DATASET` and `CONFIRM_LARGE` in the cell below, then Run All.


In [2]:
from dataclasses import dataclass, field

@dataclass
class Config:
    # Which dataset to fetch. One of: "ROG-Dialog", "ROG", "GOS", "ParlaSpeech-HR",
    # "ParlaSpeech-RS", "ParlaSpeech-CZ", "ParlaSpeech-PL".
    dataset: str = "ROG-Dialog"

    # Set True to allow files marked as `is_large=True` to actually download.
    # Default False so a curious user can't accidentally pull 50 GB.
    confirm_large: bool = True                                                           ############ Download Safety Switch

    # Force re-download even if the file already exists.
    force: bool = False

    # Skip the unpack step.
    download_only: bool = False

    # Test mode: list what would be done but don't actually fetch.
    test_mode: bool = False

cfg = Config()
print(cfg)

Config(dataset='ROG-Dialog', confirm_large=True, force=False, download_only=False, test_mode=False)


---

## 2. Dataset registry

Single source of truth for what can be downloaded. Sizes are approximate and
worth verifying before a real run.


In [3]:
DATASETS = {
    "ROG-Dialog": {
        "handle": "http://hdl.handle.net/11356/2073",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/2073",
        "files": [
            # (filename, approx_size_mb, is_large)
            ("ROG-Dialog.zip",       5,    True),   # transcriptions, annotations, metadata
            ("ROG-Dialog_audio.zip", 1220, True),    # ~1.2 GB of audio
        ],
        "notes": (
            "Dialogue corpus with sentiment and dialogue-act annotations. "
            "25 speakers, 5.2 hours, EXB/TRS/TXT formats. "
            "ROG-Dialog_audio.zip is large; required for any audio task."
        ),
    },
    "ROG": {
        "handle": "http://hdl.handle.net/11356/2062",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/2062",
        "files": [
            # (filename, approx_size_mb, is_large)
            ("ROG.zip",         30,   False),
            ("ROG-Art.wav.zip", 1400, False),   # ~1.4 GB of WAVs
        ],
        "notes": "ROG-Art.wav.zip is large; required for any audio task.",
    },
    "GOS": {
        "handle": "http://hdl.handle.net/11356/1863",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1863",
        "files": [
            ("Gos.TEI.zip",  60, False),
            ("Gos.TRS.zip",  50, False),
            ("Gos.TXT.zip",  10, False),
            ("Gos.vert.zip", 30, False),
        ],
        "notes": (
            "GOS audio is on a separate, restricted CLARIN handle "
            "(http://hdl.handle.net/11356/1973). This downloader does NOT fetch "
            "audio — request access manually."
        ),
    },
    "ParlaSpeech-HR": {
        "handle": "http://hdl.handle.net/11356/1833",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1833",
        "files": [
            ("ParlaSpeech-HR.v3.0.jsonl.gz",     800, False),
            ("ParlaSpeech-HR.v3.0.vert.gz",      400, False),
            ("ParlaSpeech-HR.v3.0.textgrid.tgz", 1500, True),
        ],
        "notes": "Croatian. Audio not in this release — see ParlaSpeech 2.0 handle for HR audio.",
    },
    "ParlaSpeech-RS": {
        "handle": "http://hdl.handle.net/11356/1833",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1833",
        "files": [
            ("ParlaSpeech-RS.v3.0.jsonl.gz",     300, False),
            ("ParlaSpeech-RS.v3.0.vert.gz",      150, False),
            ("ParlaSpeech-RS.v3.0.textgrid.tgz", 500, True),
        ],
        "notes": "Serbian.",
    },
    "ParlaSpeech-CZ": {
        "handle": "http://hdl.handle.net/11356/1833",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1833",
        "files": [
            ("ParlaSpeech-CZ.v3.0.jsonl.gz", 600, False),
            ("ParlaSpeech-CZ.v3.0.vert.gz",  300, False),
        ],
        "notes": "Czech. No TextGrid release in v3.0 (no stress/grapheme alignment).",
    },
    "ParlaSpeech-PL": {
        "handle": "http://hdl.handle.net/11356/1833",
        "base_url": "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1833",
        "files": [
            ("ParlaSpeech-PL.v3.0.jsonl.gz", 500, False),
            ("ParlaSpeech-PL.v3.0.vert.gz",  250, False),
        ],
        "notes": "Polish. No TextGrid release in v3.0.",
    },
}

print(f"Available datasets: {list(DATASETS.keys())}")
print(f"Selected:           {cfg.dataset}")
if cfg.dataset not in DATASETS:
    raise ValueError(f"Unknown dataset {cfg.dataset!r}. Choose from {list(DATASETS)}.")

Available datasets: ['ROG-Dialog', 'ROG', 'GOS', 'ParlaSpeech-HR', 'ParlaSpeech-RS', 'ParlaSpeech-CZ', 'ParlaSpeech-PL']
Selected:           ROG-Dialog


---

## 3. Plan the download

We list every file the selected dataset has, mark which are skipped (already
present, or too large without confirmation), and only then actually download.

This is the dry-run pass — nothing touches disk yet.


In [4]:
raw_dir = PROJECT_ROOT / "data" / "raw" / cfg.dataset
raw_dir.mkdir(parents=True, exist_ok=True)

spec = DATASETS[cfg.dataset]

plan = []  # list of dicts: {filename, url, dest, size_mb, is_large, action, reason}
for fname, size_mb, is_large in spec["files"]:
    dest = raw_dir / fname
    url  = f"{spec['base_url']}/{fname}"
    already = dest.exists() and dest.stat().st_size > 0

    if already and not cfg.force:
        action, reason = "skip", "already downloaded"
    elif is_large and not cfg.confirm_large:
        action, reason = "skip", f"large file ({size_mb} MB) — set confirm_large=True"
    elif cfg.test_mode:
        action, reason = "skip", "test_mode"
    else:
        action, reason = "download", "ok"

    plan.append({
        "filename": fname,
        "url":      url,
        "dest":     dest,
        "size_mb":  size_mb,
        "is_large": is_large,
        "action":   action,
        "reason":   reason,
    })

udp.banner(f"Download plan for {cfg.dataset}")
print(f"notes: {spec['notes']}\n")
for p in plan:
    flag = "📥" if p["action"] == "download" else "⏭️ "
    print(f"  {flag} {p['filename']:45s} ~{p['size_mb']:>5} MB   [{p['action']}: {p['reason']}]")

total_dl = sum(p["size_mb"] for p in plan if p["action"] == "download")
print(f"\nTotal to download: ~{total_dl} MB into {raw_dir}")



Download plan for ROG-Dialog
notes: Dialogue corpus with sentiment and dialogue-act annotations. 25 speakers, 5.2 hours, EXB/TRS/TXT formats. ROG-Dialog_audio.zip is large; required for any audio task.

  📥 ROG-Dialog.zip                                ~    5 MB   [download: ok]
  📥 ROG-Dialog_audio.zip                          ~ 1220 MB   [download: ok]

Total to download: ~1225 MB into /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/raw/ROG-Dialog


---

## 4. Download helper

A small wrapper over `requests` with a `tqdm` progress bar. Streams to a
`.part` file and renames on success — so an aborted download never leaves a
half-finished file at the final path.


In [5]:
import requests
from tqdm.auto import tqdm

def download_file(url: str, dest: Path, *, chunk_size: int = 1 << 20) -> Path:
    """
    Download `url` to `dest`. Streams via .part file for atomicity.
    Returns dest on success.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with part.open("wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, unit_divisor=1024,
            desc=dest.name, leave=True,
        ) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    part.rename(dest)
    return dest


---

## 5. Execute the plan

Loop over `plan`, download anything marked `action == "download"`. Skips are
just printed.


In [6]:
if cfg.test_mode:
    print("🧪 TEST MODE: skipping all downloads")
else:
    for p in plan:
        if p["action"] != "download":
            print(f"⏭️  {p['filename']}  ({p['reason']})")
            continue
        print(f"📥 {p['filename']}  ←  {p['url']}")
        try:
            download_file(p["url"], p["dest"])
            print(f"   ✅ wrote {p['dest']}  ({p['dest'].stat().st_size / 1e6:.1f} MB)")
        except Exception as e:
            print(f"   ❌ failed: {e}")

print("\nDone.")


📥 ROG-Dialog.zip  ←  https://www.clarin.si/repository/xmlui/bitstream/handle/11356/2073/ROG-Dialog.zip


ROG-Dialog.zip:   0%|          | 0.00/4.65M [00:00<?, ?B/s]

   ✅ wrote /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/raw/ROG-Dialog/ROG-Dialog.zip  (4.9 MB)
📥 ROG-Dialog_audio.zip  ←  https://www.clarin.si/repository/xmlui/bitstream/handle/11356/2073/ROG-Dialog_audio.zip


ROG-Dialog_audio.zip:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

   ✅ wrote /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/raw/ROG-Dialog/ROG-Dialog_audio.zip  (1305.9 MB)

Done.


---

## 6. Unpack archives

`.zip` → `unzip`, `.tgz` / `.tar.gz` → `tar`, `.gz` (single file) → `gzip -d`
into a sibling file.

Idempotent: if the unpacked directory exists and is non-empty, we skip.


In [7]:
import zipfile
import tarfile
import gzip
import shutil

unpacked_dir = PROJECT_ROOT / "data" / "unpacked" / cfg.dataset
unpacked_dir.mkdir(parents=True, exist_ok=True)

def unpack(archive: Path, dest_dir: Path) -> None:
    name = archive.name
    # Per-archive subfolder so multiple archives from one dataset don't collide.
    stem = name
    for suf in (".tar.gz", ".tgz", ".jsonl.gz", ".zip", ".gz"):
        if name.endswith(suf):
            stem = name[: -len(suf)]
            break
    out = dest_dir / stem

    if out.exists() and any(out.iterdir()):
        print(f"⏭️  {name}  already unpacked at {out}")
        return

    out.mkdir(parents=True, exist_ok=True)
    print(f"📦 unpacking {name} → {out}")

    if name.endswith(".zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(out)
    elif name.endswith((".tar.gz", ".tgz")):
        with tarfile.open(archive, "r:gz") as tf:
            tf.extractall(out)
    elif name.endswith(".jsonl.gz") or name.endswith(".gz"):
        # Single-file gzip: write the decompressed file inside `out` with the
        # .gz suffix removed.
        decomp_name = name[: -len(".gz")]
        with gzip.open(archive, "rb") as f_in, (out / decomp_name).open("wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    else:
        print(f"   ⚠️  no unpacker for {name}, skipping")
        return

    print(f"   ✅ unpacked to {out}")


if cfg.download_only:
    print("⏭️  download_only=True, not unpacking")
elif cfg.test_mode:
    print("🧪 TEST MODE: skipping unpack")
else:
    for p in plan:
        if not p["dest"].exists():
            print(f"⏭️  {p['filename']}  (not on disk, can't unpack)")
            continue
        try:
            unpack(p["dest"], unpacked_dir)
        except Exception as e:
            print(f"   ❌ unpack failed for {p['filename']}: {e}")

print("\nDone.")


📦 unpacking ROG-Dialog.zip → /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/unpacked/ROG-Dialog/ROG-Dialog
   ✅ unpacked to /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/unpacked/ROG-Dialog/ROG-Dialog
📦 unpacking ROG-Dialog_audio.zip → /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/unpacked/ROG-Dialog/ROG-Dialog_audio
   ✅ unpacked to /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/unpacked/ROG-Dialog/ROG-Dialog_audio

Done.


---

## 7. What's on disk now

A quick inventory of what we ended up with.


In [8]:
def show_tree(root: Path, max_entries: int = 30) -> None:
    if not root.exists():
        print(f"(no such dir: {root})")
        return
    entries = sorted(root.rglob("*"))
    print(f"{root}  ({len(entries)} entries)")
    for e in entries[:max_entries]:
        rel = e.relative_to(root)
        size = f" {e.stat().st_size / 1e6:.1f} MB" if e.is_file() else ""
        kind = "📁" if e.is_dir() else "📄"
        print(f"  {kind} {rel}{size}")
    if len(entries) > max_entries:
        print(f"  ... and {len(entries) - max_entries} more")

print("=== data/raw ===")
show_tree(PROJECT_ROOT / "data" / "raw" / cfg.dataset)
print()
print("=== data/unpacked ===")
show_tree(PROJECT_ROOT / "data" / "unpacked" / cfg.dataset)


=== data/raw ===
/home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/raw/ROG-Dialog  (2 entries)
  📄 ROG-Dialog.zip 4.9 MB
  📄 ROG-Dialog_audio.zip 1305.9 MB

=== data/unpacked ===
/home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline/data/unpacked/ROG-Dialog  (103 entries)
  📁 ROG-Dialog
  📁 ROG-Dialog/ROG-Dialog
  📁 ROG-Dialog/ROG-Dialog/DATA
  📁 ROG-Dialog/ROG-Dialog/DATA/EXB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0005.exb 0.7 MB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0007.exb 0.9 MB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0008.exb 0.4 MB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0009.exb 0.7 MB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0011.exb 0.4 MB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0012.exb 0.5 MB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0016.exb 0.4 MB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0018.exb 0.6 MB
  📄 ROG-Dialog/ROG-Dialog/DATA/EXB/ROG-Dia-GSO-P0019.

---

## Next

- For **ROG**: open `prep_ROG.ipynb`. It reads EXB files from `data/unpacked/ROG/...` and emits the canonical JSONL.
- For **GOS**: arrange audio access from the restricted CLARIN handle, then open `prep_GOS.ipynb`.
- For **ParlaSpeech**: open `prep_ParlaSpeech.ipynb` (v1 placeholder).

If you re-run this notebook with the same config, it'll just print skip lines and do nothing — idempotency works.
